# Sparse Sensing en UCI Hydraulic — corrida de revisión (programa #1–#6)

Refuerza el paper JCC hacia ~6.5/7 respondiendo a los 6 jueces. **Cacheable** (Drive), **paralelo** (todos los CPUs con joblib + GPU para KAN), salida con **rich/tqdm**.

- **#1** Forward-selection SOLO-FÍSICO (excluye sensores virtuales SE/CE/CP) vs todos.
- **#2** Multiseed 30 seeds + tamaños de efecto (Cliff's delta).
- **#3** Tuning UNIFORME (mismo presupuesto random-search para las 8 familias).
- **#5** Expresión simbólica KAN (2 sensores, GPU).
- **#6** Friedman + Nemenyi (diagrama Critical Difference) + medianas/IQR.

**Cómo correr:** Runtime con GPU (para #5) → *Run all*. Reanuda desde el cache en Drive si se corta. Todas las etapas pesadas usan los núcleos disponibles.

In [ ]:
# 1) Repo + dependencias
import os
if not os.path.isdir('hydraulic-sparse-pdm'):
    !git clone --depth 1 https://github.com/GastonGriott/hydraulic-sparse-pdm.git
%cd hydraulic-sparse-pdm
!git pull --ff-only -q
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, os.getcwd())
import torch; print('GPU:', torch.cuda.is_available(), '| CPUs:', os.cpu_count())

In [ ]:
# 2) Cache en Drive (resumible). Si no quieres Drive, omite esta celda.
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.environ['HYDRA_CACHE'] = '/content/drive/MyDrive/hydra_cache'
except Exception as e:
    os.environ['HYDRA_CACHE'] = 'hydra_cache'
os.makedirs(os.environ['HYDRA_CACHE'], exist_ok=True)
print('cache ->', os.environ['HYDRA_CACHE'])

In [ ]:
# 3) Datos UCI + features (255 = 17 sensores x 15 descriptores). Cacheado.
import importlib, hydra.experiments as ex; importlib.reload(ex)
from rich.console import Console; from rich.table import Table
console = Console()
X, names, targets = ex.build_xy()
console.print(f'[bold green]X[/]: {X.shape}  | targets: ' +
              ', '.join(f'{k} ({len(set(v))} clases)' for k,v in targets.items()))
console.print(f'físicos: {len(ex.PHYSICAL_SENSORS)} | virtuales: {ex.VIRTUAL_SENSORS}')

In [ ]:
# 4) #1 Forward-selection: TODOS vs SOLO-FÍSICOS (candidatos paralelizados en todos los núcleos)
from tqdm.auto import tqdm
p, data = ex.cached('sparse', v=3)
if data is None:
    bar = tqdm(total=len(targets)*2*8, desc='forward-selection (pasos)')
    data = ex.run_sparse_study(X, names, targets, on_step=lambda *_: bar.update(1)); bar.close()
    ex.save_cache(p, data)
for t, res in data.items():
    tab = Table(title=f'#1 Forward selection — {t}')
    tab.add_column('variante'); tab.add_column('orden de sensores (F1 acumulado)')
    for k in ['all','physical_only']:
        tab.add_row(k, '  ->  '.join(f'{s} ({f})' for s,f in res[k]))
    console.print(tab)
console.print('[bold]Lectura:[/] compara cuántos sensores FÍSICOS reales hacen falta vs cuando se permite el virtual SE.')

In [ ]:
# 5) #3 Tuning UNIFORME (mismo presupuesto N_ITER para todas las familias; n_jobs=-1). Cacheado por familia.
N_ITER = 80
zoo = ex.model_zoo(); tuned = {}
for t, y in targets.items():
    tuned[t] = {}
    for fam,(fac,dist) in tqdm(zoo.items(), desc=f'tuning {t}'):
        pp, d = ex.cached('tune', fam=fam, t=t, n=N_ITER)
        if d is None:
            bp, sc = ex.tune_uniform(fac, dist, X, y, n_iter=N_ITER)
            d = {'params': bp, 'cv_f1': sc}; ex.save_cache(pp, d)
        tuned[t][fam] = d
for t in targets:
    tab = Table(title=f'#3 Tuning uniforme (N={N_ITER} trials c/u) — {t}')
    tab.add_column('familia'); tab.add_column('CV macro-F1', justify='right')
    for fam,d in sorted(tuned[t].items(), key=lambda kv:-kv[1]['cv_f1']):
        tab.add_row(fam, f"{d['cv_f1']:.4f}")
    console.print(tab)

In [ ]:
# 6) #2 Multiseed 30 seeds (n_jobs=-1) con los hiperparámetros tuneados + #6 medianas/IQR
N_SEEDS = 30; perf = {}
for t, y in targets.items():
    perf[t] = {}
    for fam,(fac,_) in tqdm(zoo.items(), desc=f'multiseed {t}'):
        pp, d = ex.cached('multiseed', fam=fam, t=t, s=N_SEEDS, n=N_ITER)
        if d is None:
            d = ex.multiseed_f1(fac, tuned[t][fam]['params'], X, y, n_seeds=N_SEEDS); ex.save_cache(pp, d)
        perf[t][fam] = d
for t in targets:
    tab = Table(title=f'#2 Multiseed ({N_SEEDS} seeds) — {t}')
    for c in ['familia','mean','std','median','IQR','min-max']: tab.add_column(c)
    for fam,v in sorted(perf[t].items(), key=lambda kv:-ex.describe(kv[1])['mean']):
        s = ex.describe(v)
        tab.add_row(fam, f"{s['mean']:.4f}", f"{s['std']:.4f}", f"{s['median']:.4f}",
                    f"{s['iqr']:.4f}", f"{s['min']:.3f}-{s['max']:.3f}")
    console.print(tab)

In [ ]:
# 7) #6 Friedman + Nemenyi + diagrama Critical Difference
import matplotlib.pyplot as plt
for t in targets:
    fn = ex.friedman_nemenyi(perf[t])
    console.print(f"[bold]{t}[/]: Friedman chi2={fn['stat']:.2f} p={fn['p']:.2e}  CD={fn['cd']:.2f} (k={fn['k']}, n={fn['n']})")
    order = sorted(fn['avg_rank'].items(), key=lambda kv: kv[1])
    fig, ax = plt.subplots(figsize=(7, 0.5*len(order)+1))
    ax.barh([f for f,_ in order][::-1], [r for _,r in order][::-1], color='#2471a3')
    ax.axvline(order[0][1]+fn['cd'], ls='--', color='red', label='mejor rank + CD')
    ax.set_xlabel('rank promedio (menor=mejor)'); ax.set_title(f'Critical Difference — {t}'); ax.legend()
    plt.tight_layout(); plt.savefig(os.environ['HYDRA_CACHE']+f'/cd_{t}.png', dpi=140); plt.show()

In [ ]:
# 8) #5 Expresión simbólica KAN (2 sensores, GPU). SE+TS2 vs físicos PS1+TS2.
ksym = {}
for pair in [('SE','TS2'), ('PS1','TS2')]:
    pp, d = ex.cached('kan', pair=pair)
    if d is None:
        d = ex.kan_symbolic(X, names, targets['accum'], sensors=pair); ex.save_cache(pp, d)
    ksym['+'.join(pair)] = d
    console.print(f"[bold]KAN {pair}[/] (accum): F1={d['test_f1_macro']} [{d['device']}]")
    for i,f in enumerate(d['formulas']): console.print(f'  clase {i}: {f[:160]}')

In [ ]:
# 9) Consolidar resultados y descargar
import json
summary = {'sparse': data, 'tuned': tuned, 'multiseed': perf,
           'friedman': {t: ex.friedman_nemenyi(perf[t]) for t in targets}, 'kan_symbolic': ksym}
outp = os.environ['HYDRA_CACHE']+'/revision_results.json'
json.dump(summary, open(outp,'w'), indent=2, default=str)
console.print(f'[bold green]Listo[/] -> {outp}')
try:
    from google.colab import files; files.download(outp)
except Exception: pass